In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# Dynamically get last week's Sunday 00:00 -> this week's Sunday 00:00 (UK local time)
uk = ZoneInfo("Europe/London")
now_uk = datetime.now(uk)

# Python weekday(): Monday=0 ... Sunday=6
days_since_sunday = (now_uk.weekday() + 1) % 7
this_sunday = (now_uk - timedelta(days=days_since_sunday)).replace(hour=0, minute=0, second=0, microsecond=0)
last_sunday = this_sunday - timedelta(days=7)

from_date = last_sunday.strftime("%d/%m/%Y")
to_date   = this_sunday.strftime("%d/%m/%Y")

print(f"Automated Weekly Run -> Processing: {from_date} 00:00 to {to_date} 00:00 (UK local time)")

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")

start_local = last_sunday
end_local   = this_sunday

if end_local <= start_local:
    raise ValueError("To date must be after From date")

start_utc = start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")
end_utc   = end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")

date_time_range = f"{start_local.strftime('%d/%m/%Y %H:%M')} - {end_local.strftime('%d/%m/%Y %H:%M')}"

print(f"Week block: {date_time_range}")
print(f"UTC filter range: {start_utc} <= t < {end_utc}")

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"  # adjust to your path

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

# Single destination sheet for all 5 MPF - Elmsall - Weekly reports, "Data" tab
SHEET_ID = "1lbgAr9Lnb7_bpZc24pf_vXPcRYiIXQUNHc6zkJvIqCc"
sh = gc.open_by_key(SHEET_ID)
ws = sh.worksheet("Data")

print("Connected to:", sh.title, "-> Tab:", ws.title)

In [0]:
# Read directly from the ABFSS delta path
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))
df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

# ------------------------------------------------------------------
# Each report has a "layout" describing how it's grouped and which
# fixed-width (10-column) row shape it writes:
#
#   event_only            : Event, Qty, StdHrs, SMV, '', '', '',      Week, Date Time Range, Area
#   bonus_event_attr       : Bonus, Event, Attribute, Qty, StdHrs, SMV, '', Week, Date Time Range, Area
#   bonus_event_attr_date  : Bonus, Event, Attribute, Date, Qty, StdHrs, SMV, Week, Date Time Range, Area
#   bonus_event            : Bonus, Event, Qty, StdHrs, SMV, '', '',   Week, Date Time Range, Area
#
# "Attribute" = one row PER element of PAYLOAD_ATTRIBUTES (exploded), not the
# array joined into one string -- an event with 2 attributes contributes 2
# rows, each carrying that event's full Qty/StdHrs/SMV.
# "Bonus" = PAYLOAD_BONUSCODE.
# "Date" (Sorter 6 only) = the calendar date (UK local) the event occurred on,
# distinct from "Date Time Range" which is the whole week's range on every row.
# ------------------------------------------------------------------
all_reports = [
    {
        "name": "PiE",
        "layout": "event_only",
        "warehouse_code": "X",
        "area_codes": ('Pick', 'Pie Station'),
        "event_types": ('COUN', 'MSKU', 'PSKU', 'RSIN'),
        "area_label": "MPF - PiE",
    },
    {
        "name": "E3 Packing",
        "layout": "bonus_event_attr",
        "warehouse_code": "X",
        "area_codes": ('E3 - Packing',),
        "event_types": None,
        "area_label": "MPF - E3 Packing",
    },
    {
        "name": "Sorter 6",
        "layout": "bonus_event_attr_date",
        "warehouse_code": "X",
        "area_codes": ('Sorter 6 - Packing',),
        "event_types": None,
        "area_label": "MPF - Sorter 6 Packing",
    },
    {
        "name": "Parcel Sort",
        "layout": "event_only",
        "warehouse_code": "X",
        "area_codes": None,
        "event_types": ('ParcelSortedToSack', 'SackMappedToPosition', 'SackUnMappedFromPosition'),
        "area_label": "MPF - Parcel Sort",
    },
    {
        "name": "E3 Top Up",
        "layout": "bonus_event",
        "warehouse_code": "X",
        "area_codes": ('Topup', 'TopUp', 'PiOrQi'),
        "event_types": None,
        "area_label": "MPF - E3 Topup",
    },
]

LAYOUT_COLUMNS = {
    "event_only": [
        'PAYLOAD_EVENTTYPE', 'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
        '', '', '', 'Week', 'Date Time Range', 'Area'
    ],
    "bonus_event_attr": [
        'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE', 'Attribute',
        'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
        '', 'Week', 'Date Time Range', 'Area'
    ],
    "bonus_event_attr_date": [
        'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE', 'Attribute', 'Date',
        'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
        'Week', 'Date Time Range', 'Area'
    ],
    "bonus_event": [
        'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE',
        'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
        '', '', 'Week', 'Date Time Range', 'Area'
    ],
}

print(f"Reports configured: {len(all_reports)}")
for r in all_reports:
    print(f"  - {r['name']} [{r['layout']}]")

In [0]:
import pandas as pd

# Week number, matching the live job's formula, anchored to the start of this week block
week_val = spark.sql(f"""
  SELECT MOD(FLOOR(DATEDIFF(to_date(from_utc_timestamp(timestamp('{start_utc}'),'Europe/London')), DATE'2025-12-14')/7)+46,52) AS w
""").collect()[0]['w']

failures = []
total_rows_pushed = 0

print(f"\n=== Week: {date_time_range} ===")

# Per-layout SELECT list, GROUP BY, and (for the two attribute layouts) the
# LATERAL VIEW that explodes PAYLOAD_ATTRIBUTES into one row per element --
# OUTER so an event with an empty attributes array still yields one row
# (Attribute = NULL, filled to '' below) instead of being dropped.
LAYOUT_SELECT = {
    "event_only": {
        "select": "PAYLOAD_EVENTTYPE",
        "group_by": "1",
        "lateral_view": "",
    },
    "bonus_event_attr": {
        "select": "PAYLOAD_BONUSCODE, PAYLOAD_EVENTTYPE, Attribute",
        "group_by": "1, 2, 3",
        "lateral_view": "LATERAL VIEW OUTER explode(PAYLOAD_ATTRIBUTES) AS Attribute",
    },
    "bonus_event_attr_date": {
        "select": (
            "PAYLOAD_BONUSCODE, PAYLOAD_EVENTTYPE, Attribute, "
            "date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP), 'Europe/London'), 'dd/MM/yyyy') AS Event_Date"
        ),
        "group_by": "1, 2, 3, 4",
        "lateral_view": "LATERAL VIEW OUTER explode(PAYLOAD_ATTRIBUTES) AS Attribute",
    },
    "bonus_event": {
        "select": "PAYLOAD_BONUSCODE, PAYLOAD_EVENTTYPE",
        "group_by": "1, 2",
        "lateral_view": "",
    },
}

# All 5 reports write into the same "Data" tab, one after another, so the
# insertion point (first row where column A is blank) is tracked locally and
# advanced after each report instead of re-reading the sheet every time.
next_row = len(ws.col_values(1)) + 1

for r in all_reports:
    name = r["name"]
    columns = LAYOUT_COLUMNS[r["layout"]]
    layout_sql = LAYOUT_SELECT[r["layout"]]

    try:
        area_clause = ""
        if r["area_codes"]:
            area_list = "(" + ", ".join(f"'{a}'" for a in r["area_codes"]) + ")"
            area_clause = f"AND PAYLOAD_AREACODE IN {area_list}"

        event_clause = ""
        if r["event_types"]:
            event_list = "(" + ", ".join(f"'{e}'" for e in r["event_types"]) + ")"
            event_clause = f"AND PAYLOAD_EVENTTYPE IN {event_list}"

        query = f"""
          SELECT
            {layout_sql['select']},
            SUM(PAYLOAD_QUANTITY) AS Total_Quantity,
            SUM(PAYLOAD_SMV)      AS Total_SMV,
            SUM(PAYLOAD_SMV) / 60 AS Total_StandardHours
          FROM landing_bonus_hub_event_parsed
          {layout_sql['lateral_view']}
          WHERE TRIM(PAYLOAD_WAREHOUSECODE) = '{r["warehouse_code"]}'
            {area_clause}
            {event_clause}
            AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
            AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')
          GROUP BY {layout_sql['group_by']}
        """

        df_r = spark.sql(query).toPandas()

        # Rename the date column (if present) to the sheet's plain "Date" header
        if 'Event_Date' in df_r.columns:
            df_r = df_r.rename(columns={'Event_Date': 'Date'})

        # OUTER explode leaves Attribute as NULL when an event has no attributes
        if 'Attribute' in df_r.columns:
            df_r['Attribute'] = df_r['Attribute'].fillna('')

        if '' in columns:
            df_r[''] = ''
        df_r['Week'] = week_val
        df_r['Date Time Range'] = date_time_range
        df_r['Area'] = r['area_label']

        df_r = df_r[columns]

        if len(df_r) == 0:
            blank_row = ['(no data this window)'] + [''] * (len(columns) - 4) + [week_val, date_time_range, r['area_label']]
            df_r = pd.DataFrame([blank_row], columns=columns)

        values = df_r.astype(str).values.tolist()
        ws.update(f"A{next_row}", values, value_input_option='USER_ENTERED')
        next_row += len(values)
        total_rows_pushed += len(df_r)
        print(f"  [{name}] {len(df_r)} rows -> {ws.title} (from row {next_row - len(values)})")

    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"  [{name}] FAILED -- {error_msg}")
        failures.append({"report": name, "error": error_msg})
        continue

print(f"\n--- Weekly run summary ---")
print(f"Week processed: {date_time_range}")
print(f"Total rows pushed: {total_rows_pushed}")
print(f"Failures: {len(failures)}")
for f in failures:
    print(f"  {f['report']} | {f['error']}")